<a href="https://colab.research.google.com/github/eloosouz/crm-analysis/blob/main/Introdu%C3%A7%C3%A3o_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [29]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.tree import DecisionTreeClassifier, export_graphviz
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score, recall_score, confusion_matrix, classification_report
import subprocess
import joblib
# Get Multiple outputs in the same cell  # Obter várias saídas na mesma célula
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
# Ignore all warnings       # Ignore todos os avisos
import warnings
warnings.filterwarnings('ignore')
warnings.filterwarnings(action='ignore', category=DeprecationWarning)
pd.set_option('display.max.columns', None)
pd.set_option('display.max_rows', None)


In [30]:
# Reading the dataset      # Lendo o conjunto de dados
dc = pd.read_csv("Churn_Modelling.csv")
dc.head(5)

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15787930,Hill,760,France,Male,41,5,17990.89,1,0,1,143395.45,0
1,2,15745172,Bearce,640,Germany,Male,32,9,106715.67,3,0,0,126955.40,1
2,3,15642467,Oliveira,558,Spain,Female,33,8,0.00,1,1,0,16288.92,1
3,4,15782314,Kay,592,Spain,Male,44,8,136935.03,2,1,1,22948.52,1
4,5,15660722,H?,612,France,Male,23,9,147862.80,1,0,0,75392.05,0


In [31]:
dc.head(10)

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15787930,Hill,760,France,Male,41,5,17990.89,1,0,1,143395.45,0
1,2,15745172,Bearce,640,Germany,Male,32,9,106715.67,3,0,0,126955.40,1
2,3,15642467,Oliveira,558,Spain,Female,33,8,0.00,1,1,0,16288.92,1
3,4,15782314,Kay,592,Spain,Male,44,8,136935.03,2,1,1,22948.52,1
4,5,15660722,H?,612,France,Male,23,9,147862.80,1,0,0,75392.05,0
5,6,15699420,H?,580,France,Female,34,3,85970.34,2,0,0,148754.37,1
6,7,15711884,Mosman,635,France,Male,21,0,139689.25,2,1,0,166585.49,1
7,8,15679468,Santos,794,Germany,Female,39,1,0.00,1,1,0,2935.65,1
8,9,15752790,Pereira,592,Spain,Male,18,7,0.00,1,1,1,65809.50,0
9,10,15777086,Souza,603,Spain,Male,46,9,88968.18,2,1,1,84858.71,0


In [32]:
# Dimension of the dataset      # Dimensão do conjunto de dados
dc.shape

(2000, 14)

In [33]:
# Describe all numerical columns   # Descreva todas as colunas numéricas
dc.describe(exclude=['O'])


,RowNumber,CustomerId,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
count,2000.000000,2.000000e+03,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000
mean,1000.500000,1.570732e+07,645.798000,37.253500,4.940500,64183.370125,1.526500,0.707000,0.508000,99858.799840,0.207000
std,577.494589,6.131227e+04,95.710498,10.020501,3.140205,59259.948888,0.592008,0.455252,0.500061,57963.235866,0.405257
min,1.000000,1.560001e+07,350.000000,18.000000,0.000000,0.000000,1.000000,0.000000,0.000000,51.280000,0.000000
25%,500.750000,1.565466e+07,582.750000,30.000000,2.000000,0.000000,1.000000,0.000000,0.000000,49003.960000,0.000000
50%,1000.500000,1.570852e+07,645.500000,37.000000,5.000000,65329.160000,1.000000,1.000000,1.000000,98656.340000,0.000000
75%,1500.250000,1.575746e+07,711.000000,44.000000,8.000000,111181.050000,2.000000,1.000000,1.000000,149642.132500,0.000000
max,2000.000000,1.581497e+07,850.000000,70.000000,10.000000,224727.840000,4.000000,1.000000,1.000000,199964.440000,1.000000


In [34]:
# Describe all cartegorical columns             # Descreva todas as colunas categóricas
dc.describe(include=['O'])

,Surname,Geography,Gender
count,2000,2000,2000
unique,25,3,2
top,Kay,France,Male
freq,103,987,1109


In [35]:
# Checking number of unique customers in the dataset            # Verificando o número de clientes únicos no conjunto de dados
dc.shape[0], dc.CustomerId.nunique()

(2000, 1989)

In [36]:
# Churn value Distribution      # Distribuição do valor de churn
dc["Exited"].value_counts()

,count
Exited,
0,1586
1,414


In [37]:
dc.groupby(['Surname']).agg({'RowNumber': 'count', 'Exited': 'mean'}
    ).reset_index().sort_values(by='RowNumber', ascending=False).head()

,Surname,RowNumber,Exited
12,Kay,103,0.233010
1,Andrews,96,0.166667
5,Chu,91,0.142857
13,Mitchell,86,0.232558
4,Boni,85,0.200000


demanda.csv


In [39]:
from google.colab import files
uploaded = files.upload()  # selecione o Churn_Modelling.csv baixado


Saving demanda.csv to demanda (4).csv


In [40]:
dc = pd.read_csv("Churn_Modelling.csv")

In [41]:
dc.groupby(['Surname']).agg({'RowNumber':'count', 'Exited':'mean'}
    ).reset_index().sort_values(by='RowNumber', ascending=False).head()

,Surname,RowNumber,Exited
12,Kay,103,0.233010
1,Andrews,96,0.166667
5,Chu,91,0.142857
13,Mitchell,86,0.232558
4,Boni,85,0.200000


In [42]:
dc.groupby(['Geography']).agg({'RowNumber':'count', 'Exited':'mean'}
    ).reset_index().sort_values(by='RowNumber', ascending=False)

,Geography,RowNumber,Exited
0,France,987,0.192503
2,Spain,512,0.207031
1,Germany,501,0.235529


In [43]:
dc.head(5)

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15787930,Hill,760,France,Male,41,5,17990.89,1,0,1,143395.45,0
1,2,15745172,Bearce,640,Germany,Male,32,9,106715.67,3,0,0,126955.40,1
2,3,15642467,Oliveira,558,Spain,Female,33,8,0.00,1,1,0,16288.92,1
3,4,15782314,Kay,592,Spain,Male,44,8,136935.03,2,1,1,22948.52,1
4,5,15660722,H?,612,France,Male,23,9,147862.80,1,0,0,75392.05,0
